# NOT RAN NO TIME Experiment Q: Verbalizer EasyEnsemble

Train **five verbalizer models** using the best hyperparameters from
Experiment L, each on a **different disjoint subset of negative examples**
(all positives + 1/5 of negatives) and a **different random seed**.

At inference time, combine via **uniform averaging** and **learned stacking**
(LogisticRegression fitted on **train** probabilities), sweep thresholds on
val, and evaluate on dev.

**Approach (EasyEnsemble + Stacking):**
1. Load best hyperparameters from Exp L (`exp_L_verbalizer_best_params.json`)
2. Split negative training examples into 5 disjoint folds
3. Train 5 `PCLVerbalizer` models, each on all positives + 1 negative fold
4. Collect sigmoid probabilities from all members on **train**, val, and dev
5. Uniform ensemble: `P = mean(sigmoid(s_1), ..., sigmoid(s_5))`
6. Learned stacking: fit `LogisticRegression` on **train** member probs → predict on val/dev
7. Sweep threshold on val set for both methods, evaluate on dev set

**Why EasyEnsemble + Stacking:**
- Each member sees more balanced data → stronger per-member recall
- Different negative subsets force diverse decision boundaries
- All negatives are covered exactly once across the ensemble → no data waste
- Stacking learns optimal member weighting without leakage (fitted on train, thresholded on val, evaluated on dev)

In [1]:
import os
import sys
import random
import logging
import gc
import json

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import classification_report, f1_score
from sklearn.linear_model import LogisticRegression
from transformers import AutoTokenizer
import matplotlib.pyplot as plt

sys.path.insert(0, "..")
from utils.data import load_data
from utils.split import split_train_val
from utils.pcl_dataset import PCLVerbalizerDataset
from utils.pcl_deberta import PCLVerbalizer
from utils.optim import compute_pos_weight
from utils.training_loop import train_model

DATA_DIR = "../data"
OUT_DIR = "out"
MODEL_OUT_DIR = OUT_DIR
MODEL_NAME = "roberta-large"
MAX_LENGTH = 256
VAL_FRACTION = 0.15
BATCH_SIZE = 32
NUM_EPOCHS = 12
PATIENCE = 4
N_EVAL_STEPS = 35
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EXP_NAME = "Q_easyensemble"

# 5 ensemble members — each sees all positives + 1/5 of negatives
N_MEMBERS = 5
ENSEMBLE_SEEDS = [42, 123, 7, 2024, 314]

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s:\t%(message)s")
LOG = logging.getLogger(__name__)
LOG.info(f"Device: {DEVICE}")
os.makedirs(OUT_DIR, exist_ok=True)

if os.getlogin() == "jc4423":
    LOG.info("Running on lab machines, changing caches and model directory")
    os.environ["HF_HOME"] = "/vol/bitbucket/jc4423/.cache/huggingface"
    os.environ["TRANSFORMERS_CACHE"] = "/vol/bitbucket/jc4423/.cache/huggingface"
    os.environ["HF_DATASETS_CACHE"] = "/vol/bitbucket/jc4423/.cache/huggingface"
    os.environ["PIP_CACHE_DIR"] = "/vol/bitbucket/jc4423/.cache/pip"
    os.environ["XDG_CACHE_HOME"] = "/vol/bitbucket/jc4423/.cache"
    MODEL_OUT_DIR = "/vol/bitbucket/jc4423/models/"

/vol/bitbucket/jc4423/mlenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-04 12:07:32,310 INFO:	Device: cuda
2026-03-04 12:07:32,313 INFO:	Running on lab machines, changing caches and model directory


## 1. Data Loading & Best Params

In [2]:
# Use seed 42 for the data split (must match Exp L)
SPLIT_SEED = 42
train_df, dev_df = load_data(DATA_DIR)
train_sub_df, val_sub_df = split_train_val(train_df, val_frac=VAL_FRACTION, seed=SPLIT_SEED)
tokeniser = AutoTokenizer.from_pretrained(MODEL_NAME)
LOG.info(f"Train: {len(train_sub_df)}, Val: {len(val_sub_df)}, Dev: {len(dev_df)}")

# Load best hyperparameters from Experiment L
best_params_path = os.path.join(OUT_DIR, "exp_L_verbalizer_best_params.json")
with open(best_params_path) as f:
    BEST_PARAMS = json.load(f)
print("Best params from Exp L:")
for k, v in BEST_PARAMS.items():
    print(f"  {k}: {v}")

2026-03-04 12:07:32,462 INFO:	Train/val split: 7118 train, 1257 val (val_frac=0.15)
2026-03-04 12:07:32,463 INFO:	Train, val positive count: 675, 119
2026-03-04 12:07:32,597 INFO:	HTTP Request: HEAD https://huggingface.co/roberta-large/resolve/main/config.json "HTTP/1.1 200 OK"
2026-03-04 12:07:32,760 INFO:	HTTP Request: HEAD https://huggingface.co/roberta-large/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-03-04 12:07:32,885 INFO:	HTTP Request: GET https://huggingface.co/api/models/roberta-large/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-03-04 12:07:32,976 INFO:	HTTP Request: GET https://huggingface.co/api/models/FacebookAI/roberta-large/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-03-04 12:07:33,073 INFO:	HTTP Request: GET https://huggingface.co/api/models/roberta-large/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-03-04 12:07:33,17

Best params from Exp L:
  lr: 6.977921490278437e-06
  weight_decay: 0.000691662498760998
  head_lr_multiplier: 10
  verbalizer: true_false
  template_idx: 2
  batch_size: 32
  num_epochs: 12
  patience: 4
  best_threshold: 0.275
  warmup_fraction: 0.06
  label_smoothing: 0.0


## 2. Verbalizer & Template Setup

In [3]:
from torch.utils.data import DataLoader

# --- Verbalizer variants (same as Exp L) ---
VERBALIZER_SETS = [
    ("Yes_No",     ["Yes"],  ["No"]),
    ("yes_no",     ["yes"],  ["no"]),
    ("True_False", ["True"], ["False"]),
    ("true_false", ["true"], ["false"]),
]

TEMPLATE_OPTIONS = [
    'Is the following text patronising or condescending? "{text}" Answer: {mask}',
    'Does the author of this text sound patronising or condescending? "{text}" Answer: {mask}',
    'Is this text talking down to people? "{text}" Answer: {mask}',
    'Does the following text contain patronising language? "{text}" Answer: {mask}',
]


def words_to_first_subword_ids(words: list[str]) -> list[int]:
    """Convert words to their first subword token IDs."""
    ids = []
    for w in words:
        subtokens = tokeniser.encode(w, add_special_tokens=False)
        ids.append(subtokens[0])
        n_sub = len(subtokens)
        decoded = tokeniser.decode([subtokens[0]])
        flag = " Single token" if n_sub == 1 else f" {n_sub} subwords, using first ('{decoded}')"
        LOG.info(f"    '{w}' -> id {subtokens[0]}{flag}")
    return ids


# Resolve best verbalizer and template from Exp L params
BEST_VERBALIZER_NAME = BEST_PARAMS["verbalizer"]
BEST_TEMPLATE_IDX = BEST_PARAMS["template_idx"]
BEST_TEMPLATE = TEMPLATE_OPTIONS[BEST_TEMPLATE_IDX]

verb_set = [v for v in VERBALIZER_SETS if v[0] == BEST_VERBALIZER_NAME][0]
LOG.info(f"Best verbalizer: '{BEST_VERBALIZER_NAME}'")
POS_IDS = words_to_first_subword_ids(verb_set[1])
NEG_IDS = words_to_first_subword_ids(verb_set[2])

print(f"\nVerbalizer: {BEST_VERBALIZER_NAME}")
print(f"  pos words: {verb_set[1]} -> ids {POS_IDS}")
print(f"  neg words: {verb_set[2]} -> ids {NEG_IDS}")
print(f"Template [{BEST_TEMPLATE_IDX}]: {BEST_TEMPLATE.format(text='...', mask='[MASK]')}")


def make_verbalizer_dataloaders_from_df(
    train_member_df: pd.DataFrame,
    template: str,
) -> tuple[DataLoader, DataLoader, DataLoader]:
    """Create DataLoaders with the verbalizer template applied.
    
    Unlike the standard make_verbalizer_dataloaders, this accepts a custom
    training DataFrame (for negative subsampling) while val/dev remain fixed.
    """
    def make_ds(df):
        return PCLVerbalizerDataset(
            texts=df["text"].tolist(),
            labels=df["binary_label"].tolist(),
            max_length=MAX_LENGTH,
            tokeniser=tokeniser,
            template=template,
        )

    train_ds = make_ds(train_member_df)
    val_ds = make_ds(val_sub_df)
    dev_ds = make_ds(dev_df)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    dev_loader = DataLoader(dev_ds, batch_size=BATCH_SIZE, shuffle=False)

    return train_loader, val_loader, dev_loader

2026-03-04 12:07:33,352 INFO:	Best verbalizer: 'true_false'
2026-03-04 12:07:33,354 INFO:	    'true' -> id 29225 Single token
2026-03-04 12:07:33,354 INFO:	    'false' -> id 22303 Single token



Verbalizer: true_false
  pos words: ['true'] -> ids [29225]
  neg words: ['false'] -> ids [22303]
Template [2]: Is this text talking down to people? "..." Answer: [MASK]


## 3. Negative Subsampling (EasyEnsemble)

Split negative training examples into `N_MEMBERS` disjoint folds.
Each member trains on **all positives** + **one fold of negatives**.
This gives near-balanced training sets and forces diversity.

In [ ]:
def set_seed(seed: int):
    """Set all random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True


# Extract fixed HPs from best params
LR = BEST_PARAMS["lr"]
WEIGHT_DECAY = BEST_PARAMS["weight_decay"]
HEAD_LR_MULT = BEST_PARAMS["head_lr_multiplier"]
WARMUP_FRACTION = BEST_PARAMS.get("warmup_fraction", 0.06)
LABEL_SMOOTHING = BEST_PARAMS.get("label_smoothing", 0.0)

# --- Split negatives into N_MEMBERS disjoint folds ---
pos_df = train_sub_df[train_sub_df["binary_label"] == 1]
neg_df = train_sub_df[train_sub_df["binary_label"] == 0]

# Shuffle negatives with a fixed seed, then split into N_MEMBERS disjoint folds
neg_shuffled = neg_df.sample(frac=1.0, random_state=SPLIT_SEED).reset_index(drop=True)
fold_indices = np.array_split(np.arange(len(neg_shuffled)), N_MEMBERS)

# Build per-member training DataFrames: all positives + one negative fold
member_train_dfs: list[pd.DataFrame] = []
for idx_arr in fold_indices:
    neg_fold = neg_shuffled.iloc[idx_arr]
    member_df = pd.concat([pos_df, neg_fold], ignore_index=True)
    member_train_dfs.append(member_df)

print(f"Original training set: {len(train_sub_df)} "
      f"(pos={len(pos_df)}, neg={len(neg_df)}, ratio=1:{len(neg_df)/len(pos_df):.1f})")
print(f"\n{'Member':<8} {'Total':<8} {'Pos':<6} {'Neg':<6} {'Ratio':<10}")
print("-" * 38)
for i, mdf in enumerate(member_train_dfs):
    n_p = mdf["binary_label"].sum()
    n_n = len(mdf) - n_p
    print(f"{i+1:<8} {len(mdf):<8} {n_p:<6} {n_n:<6} 1:{n_n/n_p:.1f}")

print(f"\nFixed HPs from Exp L:")
print(f"  lr:               {LR:.2e}")
print(f"  weight_decay:     {WEIGHT_DECAY:.2e}")
print(f"  head_lr_mult:     {HEAD_LR_MULT}")
print(f"  warmup_fraction:  {WARMUP_FRACTION}")
print(f"  label_smoothing:  {LABEL_SMOOTHING}")
print(f"  verbalizer:       {BEST_VERBALIZER_NAME}")
print(f"  template_idx:     {BEST_TEMPLATE_IDX}")
print(f"  ensemble seeds:   {ENSEMBLE_SEEDS}")
print()

Original training set: 7118 (pos=675, neg=6443, ratio=1:9.5)

Member   Total    Pos    Neg    Ratio     
--------------------------------------
1        1964     675    1289   1:1.9
2        1964     675    1289   1:1.9
3        1964     675    1289   1:1.9
4        1963     675    1288   1:1.9
5        1963     675    1288   1:1.9

Fixed HPs from Exp L:
  lr:               6.98e-06
  weight_decay:     6.92e-04
  head_lr_mult:     10
  warmup_fraction:  0.06
  label_smoothing:  0.0
  verbalizer:       true_false
  template_idx:     2
  ensemble seeds:   [42, 123, 7, 2024, 314]



: 

## 4. Train Ensemble Members

Train 5 models, each on all positives + a disjoint 1/5 of negatives,
with different random seeds. Each model is saved independently.

In [ ]:
ensemble_results = []

for i, seed in enumerate(ENSEMBLE_SEEDS):
    LOG.info(f"{'='*60}")
    LOG.info(f"[{EXP_NAME}] Training member {i+1}/{N_MEMBERS} "
             f"(seed={seed}, neg_fold={i}, train_size={len(member_train_dfs[i])})")
    LOG.info(f"{'='*60}")

    set_seed(seed)

    # Each member gets its own negative fold
    train_loader, val_loader, dev_loader = make_verbalizer_dataloaders_from_df(
        member_train_dfs[i], BEST_TEMPLATE
    )

    model = PCLVerbalizer(
        pos_verbalizer_ids=POS_IDS,
        neg_verbalizer_ids=NEG_IDS,
        model_name=MODEL_NAME,
        gradient_checkpointing=True,
        cache_dir="/vol/bitbucket/jc4423/.cache/huggingface",
    ).to(DEVICE)

    # Recompute pos_weight for the subsampled training set
    pos_weight = compute_pos_weight(member_train_dfs[i], DEVICE)

    results = train_model(
        model=model, device=DEVICE,
        train_loader=train_loader, val_loader=val_loader, dev_loader=dev_loader,
        pos_weight=pos_weight, lr=LR, weight_decay=WEIGHT_DECAY,
        num_epochs=NUM_EPOCHS, warmup_fraction=WARMUP_FRACTION,
        patience=PATIENCE, head_lr_multiplier=HEAD_LR_MULT,
        label_smoothing=LABEL_SMOOTHING, eval_every_n_steps=N_EVAL_STEPS,
        use_8bit_adam=True,
    )

    # Save this member's checkpoint
    model_path = os.path.join(MODEL_OUT_DIR, f"exp_{EXP_NAME}_member_{i}_seed{seed}.pt")
    torch.save(
        {k: v.cpu() for k, v in model.state_dict().items()},
        model_path,
    )
    LOG.info(f"Saved member {i+1} to {model_path}")

    ensemble_results.append({
        "seed": seed,
        "member_idx": i,
        "train_size": len(member_train_dfs[i]),
        "best_val_f1": results["best_val_f1"],
        "best_threshold": results["best_threshold"],
        "dev_f1": results["dev_metrics"]["f1"],
        "dev_precision": results["dev_metrics"]["precision"],
        "dev_recall": results["dev_metrics"]["recall"],
        "model_path": model_path,
    })

    LOG.info(f"Member {i+1}: val F1={results['best_val_f1']:.4f}, "
             f"dev F1={results['dev_metrics']['f1']:.4f}")

    del model, train_loader, val_loader, dev_loader
    gc.collect()
    torch.cuda.empty_cache()

# Summary of individual members
print(f"\n{'Member':<8} {'Seed':<6} {'Train':<7} {'Val F1':<10} {'Dev F1':<10} {'Dev P':<10} {'Dev R':<10}")
print("-" * 61)
for r in ensemble_results:
    print(f"{r['member_idx']+1:<8} {r['seed']:<6} {r['train_size']:<7} {r['best_val_f1']:<10.4f} "
          f"{r['dev_f1']:<10.4f} {r['dev_precision']:<10.4f} {r['dev_recall']:<10.4f}")

2026-03-04 12:07:33,413 INFO:	============================================================
2026-03-04 12:07:33,414 INFO:	[Q_easyensemble] Training member 1/5 (seed=42, neg_fold=0, train_size=1964)
2026-03-04 12:07:33,414 INFO:	============================================================
2026-03-04 12:07:33,851 WARNING:	Sample 359: [MASK] token was truncated (text may be too long for max_length=256). Falling back to position 0.
2026-03-04 12:07:33,854 WARNING:	Sample 688: [MASK] token was truncated (text may be too long for max_length=256). Falling back to position 0.
2026-03-04 12:07:33,856 WARNING:	Sample 902: [MASK] token was truncated (text may be too long for max_length=256). Falling back to position 0.
2026-03-04 12:07:34,024 WARNING:	Sample 1104: [MASK] token was truncated (text may be too long for max_length=256). Falling back to position 0.
2026-03-04 12:07:34,279 WARNING:	Sample 1482: [MASK] token was truncated (text may be too long for max_length=256). Falling back to positio

## 5. Ensemble Inference

Load all 5 models, collect sigmoid probabilities on **train**, val, and dev
sets, then combine via uniform averaging and learned stacking.

In [ ]:
@torch.no_grad()
def collect_probs(
    model: PCLVerbalizer,
    device: torch.device,
    dataloader: DataLoader,
) -> tuple[np.ndarray, np.ndarray]:
    """Run inference and return (probs, labels) as numpy arrays."""
    model.eval()
    all_scores, all_labels = [], []
    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"]
        mask_token_indices = batch.get("mask_token_indices", None)
        if mask_token_indices is not None:
            mask_token_indices = mask_token_indices.to(device)

        scores = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            mask_token_indices=mask_token_indices,
        ).squeeze(-1)
        all_scores.append(scores.cpu())
        all_labels.append(labels)

    all_scores = torch.cat(all_scores)
    all_labels = torch.cat(all_labels)
    probs = torch.sigmoid(all_scores).numpy()
    labels = all_labels.long().numpy()
    return probs, labels


def find_best_threshold_from_probs(
    probs: np.ndarray, labels: np.ndarray,
) -> tuple[float, float]:
    """Sweep thresholds to find the one maximising F1."""
    best_f1, best_thresh = 0.0, 0.5
    for thresh in np.arange(0.25, 0.70, 0.025):
        preds = (probs >= thresh).astype(int)
        f1 = float(f1_score(labels, preds, zero_division=0))
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = float(thresh)
    return best_thresh, best_f1

In [ ]:
# Create deterministic dataloaders for ensemble evaluation
set_seed(SPLIT_SEED)

# Build a train loader over the full train_sub_df for stacking features
train_eval_ds = PCLVerbalizerDataset(
    texts=train_sub_df["text"].tolist(),
    labels=train_sub_df["binary_label"].tolist(),
    max_length=MAX_LENGTH,
    tokeniser=tokeniser,
    template=BEST_TEMPLATE,
)
train_eval_loader = DataLoader(train_eval_ds, batch_size=BATCH_SIZE, shuffle=False)

# Val / dev loaders (reuse helper — train_loader unused)
_, val_loader, dev_loader = make_verbalizer_dataloaders_from_df(
    member_train_dfs[0], BEST_TEMPLATE
)

# Collect probabilities from each member on train, val, dev
train_probs_list: list[np.ndarray] = []
val_probs_list: list[np.ndarray] = []
dev_probs_list: list[np.ndarray] = []
train_labels: np.ndarray = np.array([])
val_labels: np.ndarray = np.array([])
dev_labels: np.ndarray = np.array([])

for i, r in enumerate(ensemble_results):
    LOG.info(f"Loading member {i+1} from {r['model_path']}")

    model = PCLVerbalizer(
        pos_verbalizer_ids=POS_IDS,
        neg_verbalizer_ids=NEG_IDS,
        model_name=MODEL_NAME,
    ).to(DEVICE)

    state_dict = torch.load(r["model_path"], map_location=DEVICE)
    model.load_state_dict(state_dict)

    tp, tl = collect_probs(model, DEVICE, train_eval_loader)
    vp, vl = collect_probs(model, DEVICE, val_loader)
    dp, dl = collect_probs(model, DEVICE, dev_loader)

    train_probs_list.append(tp)
    val_probs_list.append(vp)
    dev_probs_list.append(dp)

    if len(train_labels) == 0:
        train_labels = tl
        val_labels = vl
        dev_labels = dl

    del model, state_dict
    gc.collect()
    torch.cuda.empty_cache()

# Stack into (N_samples, N_members) matrices
train_probs_matrix = np.column_stack(train_probs_list)  # (N_train, 5)
val_probs_matrix = np.column_stack(val_probs_list)      # (N_val, 5)
dev_probs_matrix = np.column_stack(dev_probs_list)      # (N_dev, 5)

# Uniform average
val_probs_uniform = np.mean(val_probs_matrix, axis=1)
dev_probs_uniform = np.mean(dev_probs_matrix, axis=1)

LOG.info(f"Ensemble probs collected: train={train_probs_matrix.shape}, "
         f"val={val_probs_matrix.shape}, dev={dev_probs_matrix.shape}")

## 6. Learned Stacking (fitted on train)

Fit a `LogisticRegression` on the 5-dimensional **train** probability features
(no leakage). Threshold sweep on val, evaluate on dev.

In [ ]:
# Fit LogisticRegression on TRAIN member probabilities → no leakage
stacker = LogisticRegression(
    C=1.0, max_iter=1000, solver="lbfgs", random_state=SPLIT_SEED
)
stacker.fit(train_probs_matrix, train_labels)

# Learned weights (coefficients show relative member importance)
print("Stacker coefficients (member weights):")
for i, coef in enumerate(stacker.coef_[0]):
    print(f"  Member {i+1} (seed={ENSEMBLE_SEEDS[i]}): {coef:.4f}")
print(f"  Intercept: {stacker.intercept_[0]:.4f}")

# Stacker probabilities on val/dev (NOT train — avoid leakage)
val_probs_stacked = stacker.predict_proba(val_probs_matrix)[:, 1]
dev_probs_stacked = stacker.predict_proba(dev_probs_matrix)[:, 1]

# Find best thresholds for both methods on val set
uniform_thresh, uniform_val_f1 = find_best_threshold_from_probs(val_probs_uniform, val_labels)
stacked_thresh, stacked_val_f1 = find_best_threshold_from_probs(val_probs_stacked, val_labels)

LOG.info(f"Uniform: thresh={uniform_thresh:.3f}, val F1={uniform_val_f1:.4f}")
LOG.info(f"Stacked: thresh={stacked_thresh:.3f}, val F1={stacked_val_f1:.4f}")

## 7. Results

In [ ]:
# Uniform ensemble evaluation
uniform_dev_preds = (dev_probs_uniform >= uniform_thresh).astype(int)
uniform_dev_f1 = float(f1_score(dev_labels, uniform_dev_preds, zero_division=0))

# Stacked ensemble evaluation
stacked_dev_preds = (dev_probs_stacked >= stacked_thresh).astype(int)
stacked_dev_f1 = float(f1_score(dev_labels, stacked_dev_preds, zero_division=0))

# Pick the better method for the classification report
if stacked_dev_f1 >= uniform_dev_f1:
    best_method = "Stacked (learned weights)"
    best_dev_preds = stacked_dev_preds
    best_dev_f1 = stacked_dev_f1
    best_thresh = stacked_thresh
    best_val_f1 = stacked_val_f1
else:
    best_method = "Uniform (avg prob)"
    best_dev_preds = uniform_dev_preds
    best_dev_f1 = uniform_dev_f1
    best_thresh = uniform_thresh
    best_val_f1 = uniform_val_f1

print(f"\n{'='*60}")
print(f"{EXP_NAME.upper()} — Dev Set Results ({best_method}, threshold={best_thresh:.3f})")
print(f"{'='*60}")
print(classification_report(dev_labels, best_dev_preds, target_names=["Non-PCL", "PCL"]))

# Compare all methods
print(f"\n{'Model':<28} {'Val F1':<10} {'Dev F1':<10}")
print("-" * 48)
for r in ensemble_results:
    print(f"Member {r['member_idx']+1} (seed={r['seed']}){'':<10} "
          f"{r['best_val_f1']:<10.4f} {r['dev_f1']:<10.4f}")
print(f"{'Uniform (avg prob)':<28} {uniform_val_f1:<10.4f} {uniform_dev_f1:<10.4f}")
print(f"{'Stacked (learned weights)':<28} {stacked_val_f1:<10.4f} {stacked_dev_f1:<10.4f}")

# Show config
print(f"\nConfig:")
print(f"  Verbalizer: {BEST_VERBALIZER_NAME}")
print(f"  Template [{BEST_TEMPLATE_IDX}]: \"{BEST_TEMPLATE.format(text='...', mask='[MASK]')}\"")
print(f"  lr: {LR:.2e}")
print(f"  weight_decay: {WEIGHT_DECAY:.2e}")
print(f"  head_lr_multiplier: {HEAD_LR_MULT}")
print(f"  n_members: {N_MEMBERS}")
print(f"  seeds: {ENSEMBLE_SEEDS}")
print(f"  neg subsampling: disjoint 1/{N_MEMBERS} per member")

In [ ]:
# Visualise ensemble
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1. Probability distributions per member
for i, (vp, r) in enumerate(zip(dev_probs_list, ensemble_results)):
    axes[0].hist(vp, bins=50, alpha=0.4, label=f"M{i+1} (seed={r['seed']})")
axes[0].hist(dev_probs_uniform, bins=50, alpha=0.7, color="black", histtype="step",
             linewidth=2, label="Uniform")
axes[0].axvline(uniform_thresh, color="red", linestyle="--",
               label=f"Uniform thresh={uniform_thresh:.3f}")
axes[0].set_xlabel("Probability")
axes[0].set_ylabel("Count")
axes[0].set_title("Dev Set Probability Distributions")
axes[0].legend(fontsize=7)

# 2. Stacker weights bar chart
coefs = stacker.coef_[0]
member_labels = [f"M{i+1}" for i in range(N_MEMBERS)]
bar_colors = ["steelblue" if c >= 0 else "salmon" for c in coefs]
axes[1].bar(member_labels, coefs, color=bar_colors)
axes[1].axhline(0, color="black", linewidth=0.5)
axes[1].set_ylabel("Stacker Coefficient")
axes[1].set_title("Learned Member Weights")
for j, c in enumerate(coefs):
    axes[1].text(j, c + 0.01 * np.sign(c), f"{c:.3f}", ha="center", fontsize=9)

# 3. F1 comparison: individuals vs uniform vs stacked
member_f1s = [r["dev_f1"] for r in ensemble_results]
all_labels_plot = [f"M{i+1}" for i in range(N_MEMBERS)] + ["Uniform", "Stacked"]
all_f1s = member_f1s + [uniform_dev_f1, stacked_dev_f1]
all_colours = ["steelblue"] * N_MEMBERS + ["darkorange", "forestgreen"]
axes[2].bar(range(len(all_f1s)), all_f1s, color=all_colours, tick_label=all_labels_plot)
axes[2].set_ylabel("Dev F1")
axes[2].set_title("Individual vs Ensemble F1")
axes[2].set_ylim(min(all_f1s) - 0.02, max(all_f1s) + 0.02)
for x, v in enumerate(all_f1s):
    axes[2].text(x, v + 0.003, f"{v:.4f}", ha="center", fontsize=8)

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/{EXP_NAME}_analysis.png", dpi=300)
plt.show()

In [ ]:
# Save ensemble config and results
ensemble_config = {
    "base_experiment": "L_verbalizer",
    "method": "EasyEnsemble + learned stacking (fitted on train)",
    "n_members": N_MEMBERS,
    "seeds": ENSEMBLE_SEEDS,
    "best_params": BEST_PARAMS,
    "uniform_threshold": uniform_thresh,
    "uniform_val_f1": uniform_val_f1,
    "uniform_dev_f1": uniform_dev_f1,
    "stacked_threshold": stacked_thresh,
    "stacked_val_f1": stacked_val_f1,
    "stacked_dev_f1": stacked_dev_f1,
    "stacker_coefs": stacker.coef_[0].tolist(),
    "stacker_intercept": float(stacker.intercept_[0]),
    "best_method": best_method,
    "member_results": ensemble_results,
}
config_path = os.path.join(OUT_DIR, f"exp_{EXP_NAME}_config.json")
with open(config_path, "w") as f:
    json.dump(ensemble_config, f, indent=2)
LOG.info(f"Ensemble config saved to {config_path}")